# Metal Shaders
A Metal shader is a small program written in the Metal Shading Language (MSL) that runs on the GPU, not the CPU.
It’s part of Apple’s Metal framework, which is Apple’s low-level graphics and compute API (equivalent to CUDA on NVIDIA GPUs or DirectX/Vulkan on other platforms).

A Metal shader is compiled and executed on the GPU to do things like:

- Transform and render graphics (vertex and fragment shaders for drawing)

- Run data-parallel compute workloads (compute shaders), e.g.: Image processing, Linear algebra, Physics simulation, Machine learning kernels

    - For example, if you want to add two big vectors quickly on your M4 GPU, you’d write a compute shader in Metal, compile it, and run it — rather than using CPU loops.

When you run PyTorch on MPS (torch.device("mps")), PyTorch generates and dispatches Metal compute shaders under the hood to do the math on your GPU. When you write custom shaders, you bypass PyTorch and write GPU kernels yourself — useful for custom ops, GPU microbenchmarks, or game rendering.

xcrun --find metal
> /var/run/com.apple.security.cryptexd/mnt/com.apple.MobileAsset.MetalToolchain-v17.1.324.0.Rv7DTt/Metal.xctoolchain/usr/bin/metal

xcrun --find metallib
> /var/run/com.apple.security.cryptexd/mnt/com.apple.MobileAsset.MetalToolchain-v17.1.324.0.Rv7DTt/Metal.xctoolchain/usr/bin/metallib

This is the new Cryptex-based toolchain location Apple uses on macOS Sequoia — it’s totally valid.
metal compiles .metal → .air
metallib links .air → .metallib

- Compile Metal source to AIR:

xcrun metal -c vector_add.metal -o vector_add.air


- Link to metallib:

xcrun metallib vector_add.air -o vector_add.metallib


- Dispatch from host code (C++ / Swift / Python via MPSGraph or Metal bindings), passing buffers + threadgroup size.


Apple engineers typically write:

- a minimal .metal compute shader (e.g., vector_add, strided gather, divergent loop),

- compile it to .metallib with xcrun metal + xcrun metallib,

- then write a C++ driver using metal-cpp or Swift to:

    - set up MTLDevice, MTLCommandQueue, and MTLComputePipelineState, allocate buffers of different sizes/strides,

    - encode dispatches to a command buffer and measure GPU timing using Metal’s built-in counters.

# 1) Metal Shaders (Compute / Mesh) — Source Templates

In [13]:
# Run this in a Jupyter notebook on macOS
import os, textwrap, pathlib

root = "metal_benchmarks_m4"
os.makedirs(root, exist_ok=True)

# 1. Divergent Compute Shader
divergent_compute = r"""
#include <metal_stdlib>
using namespace metal;

kernel void divergent_add(device float* dst [[buffer(0)]],
                          device const float* src [[buffer(1)]],
                          uint gid [[thread_position_in_grid]]) {
    float v = src[gid];
    if ((gid & 1) == 0) {
        for (int i=0; i<64; ++i) { v = sin(v) + cos(v); }
    } else {
        for (int i=0; i<64; ++i) { v = v * 1.0001f + 0.0001f;; }
    }
    dst[gid] = v;
}
"""
open(os.path.join(root, "divergent_compute.metal"), "w").write(divergent_compute)

# 2. Gather/Scatter Shader
gather_scatter = r"""
#include <metal_stdlib>
using namespace metal;

kernel void strided_gather(device float* dst [[buffer(0)]],
                           device const float* src [[buffer(1)]],
                           device const uint* idx [[buffer(2)]],
                           uint gid [[thread_position_in_grid]]) {
    uint index = idx[gid];
    dst[gid] = src[index];
}
"""
open(os.path.join(root, "gather_scatter.metal"), "w").write(gather_scatter)

# 3. Mesh-like Visual Shader (triangle drawn in compute)
mesh_visual = r"""
#include <metal_stdlib>
using namespace metal;

struct Triangle { float2 a, b, c; float3 color; };

inline float edge(float2 p, float2 a, float2 b) {
    float2 ab = b - a;
    float2 ap = p - a;
    return ab.x * ap.y - ab.y * ap.x;
}

kernel void mesh_visual_compute(texture2d<float, access::write> outTex [[texture(0)]],
                                constant Triangle& tri [[buffer(0)]],
                                constant uint2& size [[buffer(1)]],
                                uint2 tid [[thread_position_in_grid]]) {
    if (tid.x >= size.x || tid.y >= size.y) return;
    float2 p = (float2(tid) + 0.5) / float2(size) * 2.0 - 1.0;
    float w0 = edge(p, tri.b, tri.c);
    float w1 = edge(p, tri.c, tri.a);
    float w2 = edge(p, tri.a, tri.b);
    bool inside = (w0 >= 0.0 && w1 >= 0.0 && w2 >= 0.0) || (w0 <= 0.0 && w1 <= 0.0 && w2 <= 0.0);
    float3 col = inside ? tri.color : float3(0.02, 0.02, 0.02);
    outTex.write(float4(col, 1.0), tid);
}
"""
open(os.path.join(root, "mesh_shader_placeholder.metal"), "w").write(mesh_visual)

# 4. Host Program (Objective-C++)
host_code = r"""
// host.mm
// Build:
// clang++ -std=c++17 host.mm -o host -I./metal-cpp \
//   -framework Metal -framework Foundation -framework QuartzCore
//
// Run: ./host
// Produces out_mesh.ppm and out_ray.ppm

#define NS_PRIVATE_IMPLEMENTATION
#define CA_PRIVATE_IMPLEMENTATION
#define MTL_PRIVATE_IMPLEMENTATION

#include <Foundation/Foundation.hpp>
#include <QuartzCore/QuartzCore.hpp>
#include <Metal/Metal.hpp>

#include <vector>
#include <string>
#include <iostream>
#include <cmath>
#include <cstdio>
#include <algorithm>

static void writePPM(const std::string& path, int w, int h, const std::vector<float>& rgba) {
    FILE* f = fopen(path.c_str(), "wb");
    if (!f) { std::cerr << "Failed to open " << path << "\n"; return; }
    fprintf(f, "P6\n%d %d\n255\n", w, h);
    for (int i=0; i<w*h; i++) {
        unsigned char r = (unsigned char)std::min(255, std::max(0, int(std::lround(rgba[4*i+0]*255.f))));
        unsigned char g = (unsigned char)std::min(255, std::max(0, int(std::lround(rgba[4*i+1]*255.f))));
        unsigned char b = (unsigned char)std::min(255, std::max(0, int(std::lround(rgba[4*i+2]*255.f))));
        fputc(r, f); fputc(g, f); fputc(b, f);
    }
    fclose(f);
}

static MTL::Library* loadLib(MTL::Device* dev, const char* name) {
    NS::Error* err = nullptr;
    MTL::Library* lib = dev->newLibrary(NS::String::string(name, NS::ASCIIStringEncoding), &err);
    if (!lib) {
        std::cerr << "Failed to load " << name << (err ? std::string(": ") + err->localizedDescription()->utf8String() : "") << "\n";
    }
    return lib;
}

int main() {
    @autoreleasepool {
        auto* pool = NS::AutoreleasePool::alloc()->init();

        MTL::Device* device = MTL::CreateSystemDefaultDevice();
        if (!device) { std::cerr << "No Metal device.\n"; return -1; }

        MTL::CommandQueue* cq = device->newCommandQueue();
        if (!cq) { std::cerr << "No command queue.\n"; return -1; }

        const uint32_t W = 800, H = 600;

        // ========== MESH VISUAL (compute) ==========
        MTL::Library* meshLib = loadLib(device, "mesh_shader_placeholder.metallib");
        if (!meshLib) return -1;

        MTL::Function* meshFn  = meshLib->newFunction(NS::String::string("mesh_visual_compute", NS::ASCIIStringEncoding));
        if (!meshFn) { std::cerr << "mesh_visual_compute not found\n"; return -1; }

        NS::Error* meshErr = nullptr;
        MTL::ComputePipelineState* meshPSO = device->newComputePipelineState(meshFn, &meshErr);
        if (!meshPSO) {
            std::cerr << "mesh PSO: " << (meshErr ? meshErr->localizedDescription()->utf8String() : "(unknown)") << "\n";
            return -1;
        }

        // Render target: PRIVATE
        MTL::TextureDescriptor* tdPriv = MTL::TextureDescriptor::texture2DDescriptor(MTL::PixelFormatRGBA32Float, W, H, false);
        tdPriv->setStorageMode(MTL::StorageModePrivate);
        tdPriv->setUsage(MTL::TextureUsageShaderWrite);
        MTL::Texture* texMeshPriv = device->newTexture(tdPriv);

        // Staging target for CPU readback: SHARED
        MTL::TextureDescriptor* tdShared = MTL::TextureDescriptor::texture2DDescriptor(MTL::PixelFormatRGBA32Float, W, H, false);
        tdShared->setStorageMode(MTL::StorageModeShared);
        tdShared->setUsage(MTL::TextureUsageUnknown);
        MTL::Texture* texMeshRead = device->newTexture(tdShared);

        struct Triangle { float a[2], b[2], c[2]; float col[3]; } tri = {
            {-0.8f,-0.6f}, {0.7f,-0.4f}, {0.1f,0.8f}, {0.9f,0.3f,0.2f}
        };
        // Constant buffers: SHARED on Apple Silicon
        MTL::Buffer* bufTri = device->newBuffer(&tri, sizeof(tri), MTL::ResourceStorageModeShared);
        uint32_t size[2] = {W,H};
        MTL::Buffer* bufSize = device->newBuffer(&size, sizeof(size), MTL::ResourceStorageModeShared);

        // Dispatch compute
        {
            MTL::CommandBuffer* cb = cq->commandBuffer();
            MTL::ComputeCommandEncoder* enc = cb->computeCommandEncoder();
            enc->setComputePipelineState(meshPSO);
            enc->setTexture(texMeshPriv, 0);
            enc->setBuffer(bufTri, 0, 0);
            enc->setBuffer(bufSize, 0, 1);

            MTL::Size tg(16, 16, 1);
            MTL::Size grid(W, H, 1);
            enc->dispatchThreads(grid, tg);
            enc->endEncoding();
            cb->commit();
            cb->waitUntilCompleted();
        }

        // Blit PRIVATE -> SHARED, then read
        {
            MTL::CommandBuffer* cb = cq->commandBuffer();
            MTL::BlitCommandEncoder* blit = cb->blitCommandEncoder();
            blit->copyFromTexture(texMeshPriv, 0, 0, MTL::Origin(0,0,0), MTL::Size(W,H,1),
                                  texMeshRead, 0, 0, MTL::Origin(0,0,0));
            blit->endEncoding();
            cb->commit();
            cb->waitUntilCompleted();
        }

        std::vector<float> pixels(W*H*4);
        MTL::Region region(0,0,W,H);
        texMeshRead->getBytes(pixels.data(), W*4*sizeof(float), region, 0);
        writePPM("out_mesh.ppm", W, H, pixels);
        std::cout << "Wrote out_mesh.ppm\n";

        pool->release();
        return 0;
    }
}

"""
open(os.path.join(root, "host.mm"), "w").write(host_code)


print(f"Project created at {os.path.abspath(root)}")

Project created at /Users/hafsahshahzad/Desktop/MyDocs/TheCompilerLab/TheCompilerLab/Performance Profiling/metal_benchmarks_m4


# RUN INSTRUCTIONS

cd metal_benchmarks_m4

### Compile shaders
### Make sure you put -c in xcrun metal command. Else it generates a metallib executable. With -c flag it produces LLVM bitcode, wrapper. This can be compiled with xcrun metallib..

xcrun metal -c divergent_compute.metal -o divergent_compute.air
xcrun metallib divergent_compute.air -o divergent_compute.metallib

xcrun metal -c gather_scatter.metal -o gather_scatter.air
xcrun metallib gather_scatter.air -o gather_scatter.metallib

xcrun metal -c mesh_shader_placeholder.metal -o mesh_shader_placeholder.air
xcrun metallib mesh_shader_placeholder.air -o mesh_shader_placeholder.metallib

xcrun metal -c raytracing_placeholder.metal -o raytracing_placeholder.air
xcrun metallib raytracing_placeholder.air -o raytracing_placeholder.metallib

### Build host
### MAke sure you download metal-cpp relevant to your mac and then place the folder in same folder as rest of your files
### Make sure you include the following headers in your host.mm file:
#define NS_PRIVATE_IMPLEMENTATION
#define CA_PRIVATE_IMPLEMENTATION
#define MTL_PRIVATE_IMPLEMENTATION

#include <Foundation/Foundation.hpp>
#include <QuartzCore/QuartzCore.hpp>
#include <Metal/Metal.hpp>

clang++ -std=c++17 host.mm -o host -I./metal-cpp -framework Metal -framework Foundation -framework QuartzCore

### Run
./host
magick out_mesh.ppm out_mesh.png
magick out_ray.ppm out_ray.png
    
//open out_mesh.ppm
//open out_ray.ppm